In [1]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

matplotlib version: 3.10.0
torch version: 2.5.1
tiktoken version: 0.9.0


In [3]:
import torch
from torch import nn
import torch.nn.functional as F
import tiktoken

In [31]:
torch.set_printoptions(sci_mode=False)
torch.manual_seed(123)

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

# Layer Normalization

In [17]:
class LayereNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / (std + self.eps)
        return self.scale * norm_x + self.shift

In [25]:
# test LayerNorm
test = LayereNorm(10)
x = torch.randn(2, 10)
(test(x)).mean(), (test(x)).std()


(tensor(    -0.0000, grad_fn=<MeanBackward0>),
 tensor(1.0260, grad_fn=<StdBackward0>))

# Feed Forward

In [24]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], cfg["emb_dim"] * 4),
            nn.GELU(),
            nn.Linear(cfg["emb_dim"] * 4, cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [27]:
# test feed forward
ff = FeedForward(GPT_CONFIG_124M)
x = torch.randn(2, 3, 768)
ff(x).shape, ff(x).mean(), ff(x).std()

(torch.Size([2, 3, 768]),
 tensor(-0.0006, grad_fn=<MeanBackward0>),
 tensor(0.2050, grad_fn=<StdBackward0>))

# Multi-head Attention

In [28]:
# from previous chapter
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length=6, dropout=0.0, num_heads=2, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.out_proj = nn.Linear(d_out, d_out) # d_out*d_out + d_out(bias) trainable parameters
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        
        attention_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]
        attention_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = F.softmax(attention_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous()
        context_vec = context_vec.view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

# Transformer

In [30]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"], cfg["drop_rate"])
        self.norm1 = LayereNorm(cfg["emb_dim"])
        self.ff = FeedForward(cfg)
        self.norm2 = LayereNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        
    def forward(self, x):
        # shortcut for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        
        # shortcut for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x
        

In [32]:
x = torch.randn(2, 3, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)
print("Transformer block input shape:", x.shape)
print("Transformer block output shape:", output.shape)

Transformer block input shape: torch.Size([2, 3, 768])
Transformer block output shape: torch.Size([2, 3, 768])


# GPT model block

In [45]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emd = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emd = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emd = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayereNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emd(in_idx)
        pos_emb = self.pos_emd(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.drop_emd(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logis = self.out_head(x)
        return logis

In [46]:
# test GPT model
model = GPTModel(GPT_CONFIG_124M)
batch = torch.randint(0, 50257, (2, 768))
#model.cfg = GPT_CONFIG_124M 
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[ 5451,   847, 42242,  ..., 31995, 13721,  4628],
        [44678, 49590, 45278,  ..., 45866, 26378,  9648]])

Output shape: torch.Size([2, 768, 50257])
tensor([[[-0.4293,  0.5971,  0.1088,  ...,  1.4335, -0.4002, -0.3166],
         [ 0.2995,  0.4737, -0.6391,  ...,  0.0939,  0.2263, -0.8335],
         [-0.2331,  0.1565,  0.1996,  ..., -0.1096,  0.0641,  0.2694],
         ...,
         [ 0.2841, -0.8455, -0.1510,  ...,  0.3820, -0.2217,  0.7037],
         [ 0.7808,  1.1677,  0.3897,  ...,  0.3265,  0.3221, -0.0913],
         [ 0.3983,  0.2207,  0.6431,  ..., -0.1670, -0.0066,  0.5151]],

        [[ 0.2130, -0.9009,  0.0788,  ...,  0.6971,  0.2153, -0.2391],
         [ 0.4394,  0.3159, -0.5746,  ..., -0.3472,  1.0150,  0.0084],
         [-0.4389, -0.5999,  0.7155,  ..., -0.0791,  0.1014, -0.2308],
         ...,
         [ 0.4425, -0.3122,  0.0685,  ...,  0.5148,  0.1661,  0.5589],
         [ 1.0081,  0.3717, -0.3361,  ...,  0.6810, -0.5071, -0.3991],
         [-0.17

In [47]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536
